In [ ]:
!pip install ucimlrepo xgboost shap lightgbm

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

# ── CHARGER LA DATA ──────────────────────────────────
dataset = fetch_ucirepo(id=544)
df_original = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df = df_original.copy()

# ── ENCODAGE ─────────────────────────────────────────
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

le = LabelEncoder()
for col in ['Gender', 'CAEC', 'CALC', 'MTRANS']:
    df[col] = le.fit_transform(df[col])

le_target = LabelEncoder()
df['NObeyesdad'] = le_target.fit_transform(df['NObeyesdad'])

# ── OPTIMISATION MÉMOIRE ─────────────────────────────
def optimize_memory(df):
    before = df.memory_usage(deep=True).sum() / 1024
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    after = df.memory_usage(deep=True).sum() / 1024
    print(f"Mémoire : {before:.1f} KB → {after:.1f} KB (Gagné {((before-after)/before*100):.1f}%)")
    return df

df = optimize_memory(df)

# ── SPLIT ────────────────────────────────────────────
X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print("✅ Data prête !")

In [ ]:
from xgboost import XGBClassifier

# ── ENTRAÎNEMENT ─────────────────────────────────────
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train)
print("✅ Modèle entraîné !")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns

# ── PRÉDICTIONS ──────────────────────────────────────
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# Binariser pour ROC-AUC multiclass
classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=classes)

# ── MÉTRIQUES ────────────────────────────────────────
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')
roc_auc   = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='weighted')

# ── RÉSULTATS ────────────────────────────────────────
print("=" * 50)
print("       ÉVALUATION — XGBoost Model")
print("=" * 50)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy:.2%})")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("=" * 50)

# ── RAPPORT DÉTAILLÉ ─────────────────────────────────
print("\nRapport détaillé par classe :")
class_names = le_target.classes_
print(classification_report(y_test, y_pred, target_names=class_names))

# ── CONFUSION MATRIX ─────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — XGBoost')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ── CONCLUSION ───────────────────────────────────────
print("=" * 50)
print("CONCLUSION — Pourquoi XGBoost ?")
print("=" * 50)
print(f"""
- Accuracy  : {accuracy:.2%} → excellente performance
- ROC-AUC   : {roc_auc:.4f} → proche de 1 = excellent
- F1-Score  : {f1:.4f} → bon équilibre précision/recall
- Gère bien les 7 classes d'obésité
- Robuste aux outliers conservés
- Compatible avec SHAP pour l'explicabilité
→ XGBoost est le modèle retenu ✅
""")

In [ ]:
import shap

# ── SHAP ─────────────────────────────────────────────
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Graphique 1 : Features les plus importantes
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type='bar', show=True)

# Graphique 2 : Impact détaillé
plt.figure()
shap.summary_plot(shap_values, X_test, show=True)

In [ ]:
import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

from google.colab import files
files.download('model.pkl')
print('✅ Modèle téléchargé !')